In [4]:
import ast
import re
import pandas as pd
import numpy as np


def prepare_item_relationship_sankey(
    rules_df: pd.DataFrame,
    antecedent_col: str = "antecedent",
    consequent_col: str = "consequent",
    support_col: str = "support",
    confidence_col: str = "confidence",
    lift_col: str = "lift",
    top_n_edges: int = 30,
    split_rule_weight_across_pairs: bool = True,
):
    """
    Convert association rules dataframe into Plotly-sankey-ready data.

    Parameters
    ----------
    rules_df : pd.DataFrame
        Must contain antecedent, consequent, support, confidence, lift.
        antecedent/consequent can be either:
        - python lists
        - strings like "['A', 'B']"
    antecedent_col, consequent_col, support_col, confidence_col, lift_col : str
        Column names in rules_df.
    top_n_edges : int
        Keep only strongest item-to-item edges.
    split_rule_weight_across_pairs : bool
        If True, a rule's support is divided across all antecedent->consequent item pairs.
        This avoids overweighting rules with many items.

    Returns
    -------
    dict with:
        - nodes: list of {"name": item}
        - links: list of sankey links
        - sankey_data: dict directly usable in Plotly Sankey
        - edge_table: aggregated item-to-item edge dataframe
    """

    required = [antecedent_col, consequent_col, support_col, confidence_col, lift_col]
    missing = [c for c in required if c not in rules_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    def _clean_text(x):
        return re.sub(r"\s+", " ", str(x).strip())

    def _parse_items(value):
        if pd.isna(value):
            return []
        if isinstance(value, list):
            return [_clean_text(v) for v in value if _clean_text(v)]
        if isinstance(value, (set, tuple)):
            return [_clean_text(v) for v in list(value) if _clean_text(v)]

        raw = _clean_text(value)
        if not raw:
            return []

        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, (list, tuple, set)):
                return [_clean_text(v) for v in parsed if _clean_text(v)]
            return [_clean_text(parsed)]
        except Exception:
            raw = raw.strip("[]")
            parts = [p.strip().strip("'").strip('"') for p in raw.split(",")]
            return [_clean_text(p) for p in parts if _clean_text(p)]

    df = rules_df.copy()
    df["__ants__"] = df[antecedent_col].apply(_parse_items)
    df["__cons__"] = df[consequent_col].apply(_parse_items)

    df[support_col] = pd.to_numeric(df[support_col], errors="coerce")
    df[confidence_col] = pd.to_numeric(df[confidence_col], errors="coerce")
    df[lift_col] = pd.to_numeric(df[lift_col], errors="coerce")

    df = df.dropna(subset=[support_col, confidence_col, lift_col])
    df = df[(df["__ants__"].map(len) > 0) & (df["__cons__"].map(len) > 0)].copy()

    edge_rows = []
    for _, row in df.iterrows():
        ants = row["__ants__"]
        cons = row["__cons__"]

        pair_count = max(len(ants) * len(cons), 1)
        edge_support = row[support_col] / pair_count if split_rule_weight_across_pairs else row[support_col]

        for a in ants:
            for c in cons:
                edge_rows.append(
                    {
                        "source": a,
                        "target": c,
                        "edge_support": edge_support,
                        "confidence": row[confidence_col],
                        "lift": row[lift_col],
                    }
                )

    edge_table = pd.DataFrame(edge_rows)

    if edge_table.empty:
        return {
            "nodes": [],
            "links": [],
            "sankey_data": {"node": {"label": []}, "link": {"source": [], "target": [], "value": []}},
            "edge_table": edge_table,
        }

    edge_table = (
        edge_table.groupby(["source", "target"], as_index=False)
        .agg(
            value=("edge_support", "sum"),
            avg_confidence=("confidence", "mean"),
            avg_lift=("lift", "mean"),
            edge_count=("source", "count"),
        )
    )

    edge_table["strength_score"] = (
        0.5 * (edge_table["value"] / edge_table["value"].max())
        + 0.25 * (edge_table["avg_confidence"] / edge_table["avg_confidence"].max())
        + 0.25 * (edge_table["avg_lift"] / edge_table["avg_lift"].max())
    )

    edge_table = edge_table.sort_values(
        ["strength_score", "value", "avg_lift"],
        ascending=False
    ).head(top_n_edges).reset_index(drop=True)

    node_names = pd.unique(edge_table[["source", "target"]].values.ravel("K")).tolist()
    node_index = {name: i for i, name in enumerate(node_names)}

    nodes = [{"name": n} for n in node_names]
    links = []
    for _, row in edge_table.iterrows():
        links.append(
            {
                "source": node_index[row["source"]],
                "target": node_index[row["target"]],
                "value": float(row["value"]),
                "source_name": row["source"],
                "target_name": row["target"],
                "avg_confidence": float(row["avg_confidence"]),
                "avg_lift": float(row["avg_lift"]),
                "edge_count": int(row["edge_count"]),
                "strength_score": float(row["strength_score"]),
            }
        )

    sankey_data = {
        "node": {
            "label": [n["name"] for n in nodes]
        },
        "link": {
            "source": [l["source"] for l in links],
            "target": [l["target"] for l in links],
            "value": [l["value"] for l in links],
            "customdata": [
                [
                    l["source_name"],
                    l["target_name"],
                    l["avg_confidence"],
                    l["avg_lift"],
                    l["edge_count"],
                    l["strength_score"],
                ]
                for l in links
            ],
        },
    }

    return {
        "nodes": nodes,
        "links": links,
        "sankey_data": sankey_data,
        "edge_table": edge_table,
    }

rules_df = pd.read_csv(r"C:\Users\Alber\Desktop\uni\GraduationProject\mintel\_models\mba\association_rules.csv")
result = prepare_item_relationship_sankey(rules_df, top_n_edges=30)

# Plotly usage
import plotly.graph_objects as go

fig = go.Figure(
    go.Sankey(
        node=dict(
            pad=15,
            thickness=18,
            label=result["sankey_data"]["node"]["label"]
        ),
        link=dict(
            source=result["sankey_data"]["link"]["source"],
            target=result["sankey_data"]["link"]["target"],
            value=result["sankey_data"]["link"]["value"],
            customdata=result["sankey_data"]["link"]["customdata"],
            hovertemplate=(
                "%{customdata[0]} -> %{customdata[1]}"
                "<br>Support Weight: %{value:.6f}"
                "<br>Avg Confidence: %{customdata[2]:.3f}"
                "<br>Avg Lift: %{customdata[3]:.2f}"
                "<br>Rule Count: %{customdata[4]}"
                "<br>Strength Score: %{customdata[5]:.3f}"
                "<extra></extra>"
            ),
        ),
    )
)

fig.update_layout(title="Item Relationship Flow", font_size=11)
fig.show()

